In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from src.har.data import load
d = load()
print(d.features.shape)

(10299, 561)


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import KFold, GroupKFold, cross_val_score, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}
splits = {
    "Random (KFold)": KFold(n_splits=5, shuffle=True, random_state=42),
    "By person (GroupKFold)": GroupKFold(n_splits=5),
}

rows = []
for model_name, model in models.items():
    for split_name, cv in splits.items():
        res = cross_validate(model, d.features, d.y, cv=cv, groups=d.subject,
                             scoring=["accuracy", "f1_macro"])
        rows.append({
            "Model": model_name,
            "Split": split_name,
            "Accuracy": f"{res['test_accuracy'].mean():.3f} ± {res['test_accuracy'].std():.3f}",
            "Macro-F1": f"{res['test_f1_macro'].mean():.3f} ± {res['test_f1_macro'].std():.3f}",
        })

results = pd.DataFrame(rows)
results

c:\Users\Karl\Documents\projects\wearable-activity-recognition\.venv\Lib\site-packages\sklearn\model_selection\_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(
c:\Users\Karl\Documents\projects\wearable-activity-recognition\.venv\Lib\site-packages\sklearn\model_selection\_split.py:86: UserWarning: The groups parameter is ignored by KFold
  warnings.warn(


,Model,Split,Accuracy,Macro-F1
0,Majority baseline,Random (KFold),0.184 ± 0.011,0.052 ± 0.003
1,Majority baseline,By person (GroupKFold),0.189 ± 0.011,0.053 ± 0.003
2,Logistic regression,Random (KFold),0.984 ± 0.004,0.985 ± 0.003
3,Logistic regression,By person (GroupKFold),0.958 ± 0.014,0.957 ± 0.015


Tested on unseen people, logistic regression drops from 0.985 to 0.957 macro-F1, and fold-to-fold spread grows 4×. Random splits leak because the same person and overlapping windows land on both sides. All later models use the person split.